*AI-generated draft (Claude, Anthropic) — for review. The labeling UI is version-controlled; the corrected boxes are your own ground truth. Click-added boxes use the typical worm size (per-frame median) — an approximation adequate for training recall.*

<span style="font-family: 'Courier New', monospace;">

# 33 · Monday manual series — box-corrector (unified counts + training labels)

Corrects the **v2/v3 pre-labels** for the weekly-Monday Scene-1 series
(`validation/monday_manual_series/`). Each corrected frame is **both** a point in the
manual abundance timeseries **and** a training label — `worm_count = number of boxes`,
never typed independently (schema `notebooks/manual_series_schema.md`).

**Kernel:** `joseph-scaleworm-thesis` (needs `ipympl`; the `scaleworm-student-lab` venv does not have it).

**Routing already applied:** clear frames (before 2023-08-10) were pre-labeled by **v2**
(~12.8 boxes/frame — well seeded); blurry frames (after) by **v3** (~3.9 boxes/frame — sparse,
**you add most worms yourself**). The title bar shows which model and the train/val split.

**How to use (single clicks)**
1. **Left-click each missed worm** → drops a median-sized box on it.
2. **Right-click** a box to delete it (rare false pre-box or misclick).
3. **Undo** / **Clear** for the last box / all boxes.
4. **Save & Next ▶** → writes YOLO boxes back to `labels/<frame>.txt` and sets the manifest row to
   `frame_status=counted`, `worm_count=len(boxes)`, `counter`, `date_counted`.
5. **Unusable (blur)** / **No Scene-1** → marks the frame missing (`worm_count = NA`, **not 0** —
   no-observation is not an observed zero) and advances. **Skip** advances without changing anything.

Resumable: opens on the first frame not yet `counted`. Correct in **both** directions —
add worms the model missed, don't only delete — especially on the v3/blurry frames.
</span>

In [1]:
%matplotlib widget
import csv
import shutil
import statistics
from datetime import date
from pathlib import Path

import ipywidgets as widgets
import matplotlib.image as mpimg
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection
from IPython.display import display

REPO = Path("/home/jovyan/scaleworm-student-lab")
BASE = REPO / "validation/monday_manual_series"
IMAGES = BASE / "images"
LABELS = BASE / "labels"
MANIFEST = BASE / "frame_manifest.csv"

COUNTER = "MJ"                 # annotator initials -> manifest `counter`
TODAY = date.today().isoformat()
DEFAULT_W, DEFAULT_H = 0.0342, 0.0585  # median normalized worm box (fallback)

# frame_status values that count as "done" for resume / progress.
DONE = {"counted", "unusable_blur", "no_scene1"}


def read_manifest():
    with MANIFEST.open(newline="") as fh:
        r = csv.DictReader(fh)
        return list(r), r.fieldnames


def write_manifest(rows, fields):
    with MANIFEST.open("w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=fields)
        w.writeheader()
        w.writerows(rows)


def update_row(frame_id, **updates):
    rows, fields = read_manifest()
    for row in rows:
        if row["frame_id"] == frame_id:
            row.update(updates)
            break
    write_manifest(rows, fields)


def load_yolo(path, W, H):
    boxes = []
    if path.exists():
        for line in path.read_text().splitlines():
            p = line.split()
            if len(p) == 5:
                cx, cy, w, h = (float(v) for v in p[1:])
                cx, cy, w, h = cx * W, cy * H, w * W, h * H
                boxes.append([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2])
    return boxes


def save_yolo(path, boxes, W, H):
    lines = []
    for x1, y1, x2, y2 in boxes:
        cx, cy = (x1 + x2) / 2 / W, (y1 + y2) / 2 / H
        w, h = abs(x2 - x1) / W, abs(y2 - y1) / H
        lines.append(f"0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
    path.write_text("\n".join(lines) + ("\n" if lines else ""))


ROWS, _FIELDS = read_manifest()
INFO = {r["frame_id"]: r for r in ROWS}          # frame_id -> manifest row (status/model/split)
stems = [r["frame_id"] for r in ROWS]            # chronological (manifest order)
n_done = sum(1 for r in ROWS if r["frame_status"] in DONE)
print(f"{len(stems)} frames in the Monday series  ({n_done} already counted/marked, "
      f"{len(stems) - n_done} remaining).")
print("Left-click each missed worm; right-click a box to delete; then Save & Next.")

367 frames in the Monday series  (0 already counted/marked, 367 remaining).
Left-click each missed worm; right-click a box to delete; then Save & Next.


In [2]:
class MondayLabeler:
    """Box-correct the Monday-series pre-labels; derive worm_count from the boxes."""

    def __init__(self, stems):
        self.stems = stems
        self.pos = next(
            (i for i, s in enumerate(stems) if INFO[s]["frame_status"] not in DONE), 0
        )
        self.boxes = []
        self.coll = None
        self.bw, self.bh = DEFAULT_W, DEFAULT_H

        def mk(desc, style=""):
            return widgets.Button(description=desc, button_style=style,
                                  layout=widgets.Layout(width="auto"))

        self.b_undo = mk("Undo")
        self.b_clear = mk("Clear")
        self.b_prev = mk("◀ Prev")
        self.b_save = mk("Save & Next ▶", "success")
        self.b_blur = mk("Unusable (blur)", "danger")
        self.b_nos1 = mk("No Scene-1", "warning")
        self.b_skip = mk("Skip")
        self.b_undo.on_click(lambda _: self._undo())
        self.b_clear.on_click(lambda _: self._clear())
        self.b_prev.on_click(lambda _: self._prev())
        self.b_save.on_click(lambda _: self._save())
        self.b_blur.on_click(lambda _: self._mark("unusable_blur"))
        self.b_nos1.on_click(lambda _: self._mark("no_scene1"))
        self.b_skip.on_click(lambda _: self._advance())
        self.status = widgets.HTML()
        self.msg = widgets.Output()

        plt.ioff()
        self.fig, self.ax = plt.subplots(figsize=(12, 7))
        plt.ion()
        self.fig.canvas.header_visible = False
        self.fig.canvas.toolbar_position = "right"
        self.fig.canvas.mpl_connect("button_press_event", self._on_click)

        controls = widgets.HBox([self.b_undo, self.b_clear, self.b_prev,
                                 self.b_save, self.b_blur, self.b_nos1, self.b_skip])
        self.box = widgets.VBox([controls, self.status, self.fig.canvas, self.msg])
        self._load()

    def _stem(self):
        return self.stems[self.pos]

    def _load(self):
        stem = self._stem()
        img = mpimg.imread(IMAGES / f"{stem}.png")
        self.H, self.W = img.shape[:2]
        self.ax.clear()
        self.coll = None
        self.ax.imshow(img)
        self.ax.set_xticks([])
        self.ax.set_yticks([])
        self.boxes = load_yolo(LABELS / f"{stem}.txt", self.W, self.H)
        if len(self.boxes) >= 3:
            self.bw = statistics.median((x2 - x1) / self.W for x1, _, x2, _ in self.boxes)
            self.bh = statistics.median((y2 - y1) / self.H for _, y1, _, y2 in self.boxes)
        else:
            self.bw, self.bh = DEFAULT_W, DEFAULT_H
        self._draw()
        self._status()

    def _draw(self):
        if self.coll is not None:
            try:
                self.coll.remove()
            except ValueError:
                pass
        rects = [mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1)
                 for x1, y1, x2, y2 in self.boxes]
        self.coll = PatchCollection(rects, facecolor="none",
                                    edgecolor="#D55E00", linewidths=1.6)
        self.ax.add_collection(self.coll)
        info = INFO[self._stem()]
        self.ax.set_title(
            f"{self._stem()}  [{info['prelabel_model']} / {info['split']}]"
            f"   —   {len(self.boxes)} boxes", fontsize=10)
        self.fig.canvas.draw_idle()

    def _on_click(self, event):
        if event.inaxes != self.ax or event.xdata is None:
            return
        if getattr(self.fig.canvas, "toolbar", None) and self.fig.canvas.toolbar.mode != "":
            return  # zoom/pan active
        if event.button == 1:
            w, h = self.bw * self.W, self.bh * self.H
            cx, cy = event.xdata, event.ydata
            self.boxes.append([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2])
            self._draw()
        elif event.button == 3:
            hit = [i for i, (x1, y1, x2, y2) in enumerate(self.boxes)
                   if x1 <= event.xdata <= x2 and y1 <= event.ydata <= y2]
            if hit:
                self.boxes.pop(hit[-1])
                self._draw()

    def _undo(self):
        if self.boxes:
            self.boxes.pop()
            self._draw()

    def _clear(self):
        self.boxes = []
        self._draw()

    def _status(self):
        n_done = sum(1 for s in self.stems if INFO[s]["frame_status"] in DONE)
        st = INFO[self._stem()]["frame_status"]
        self.status.value = (
            f"<b>Frame {self.pos + 1}/{len(self.stems)}</b> &nbsp; {self._stem()} "
            f"&nbsp;<i>(status: {st})</i> &nbsp;&mdash;&nbsp; "
            f"<b>{n_done}</b> done, <b>{len(self.stems) - n_done}</b> left")

    def _save(self):
        stem = self._stem()
        save_yolo(LABELS / f"{stem}.txt", self.boxes, self.W, self.H)
        update_row(stem, frame_status="counted", worm_count=str(len(self.boxes)),
                   label_path=f"labels/{stem}.txt", annotation_method="box_corrected",
                   prelabel_model=INFO[stem]["prelabel_model"],
                   counter=COUNTER, date_counted=TODAY)
        INFO[stem]["frame_status"] = "counted"
        INFO[stem]["worm_count"] = str(len(self.boxes))
        self._advance()

    def _mark(self, status):
        # No-observation != zero: worm_count = NA (blank), frame excluded from the series.
        stem = self._stem()
        update_row(stem, frame_status=status, worm_count="",
                   annotation_method="box_corrected",
                   counter=COUNTER, date_counted=TODAY)
        INFO[stem]["frame_status"] = status
        INFO[stem]["worm_count"] = ""
        self._advance()

    def _advance(self):
        self.msg.clear_output()
        if self.pos < len(self.stems) - 1:
            self.pos += 1
            self._load()
        else:
            with self.msg:
                print("✅ Reached the last frame. Build the timeseries from the "
                      "counted rows (group by Monday, mean ± SEM over the 8 slots).")

    def _prev(self):
        if self.pos > 0:
            self.pos -= 1
            self._load()


app = MondayLabeler(stems)
display(app.box)